# Comparação dos algoritmos

Este notebook reúne os resultados dos quatro notebooks de modelagem (`modelagem-mlp`, `modelagem-nb`, `modelagem-xgb` e `modelagem-rl`) e compara os algoritmos entre si. Ele **não treina modelos**: só lê o que cada notebook gravou em `data/results/modelagem/`. Por isso, deve ser executado **depois** dos quatro.

## Estratégia de avaliação

- **Divisão treino/teste**: validação cruzada estratificada com 10 folds (`StratifiedKFold(n_splits=10, shuffle=True, random_state=42)`). Cada cliente aparece no teste exatamente uma vez. Cada fold de teste tem 285 clientes e 44 ou 45 churns (15,4% a 15,8%), a mesma proporção da base (15,6%).
- **Mesmos folds para todos**: as partições foram geradas uma única vez e gravadas em `data/processed/folds.csv`. Todos os algoritmos, todas as configurações e as duas bases (com e sem `inativo`) usam exatamente os mesmos pares treino/teste. Por isso as comparações podem ser **pareadas fold a fold**: a diferença entre dois modelos em cada fold elimina a variação causada pela dificuldade de cada partição.
- **Sem vazamento**: transformações linha a linha (remoção de duplicatas, recodificação, one-hot, `log1p`, idade numérica) são feitas na base inteira, porque não aprendem nada com os dados. As que aprendem parâmetros (padronização da MLP e da regressão logística, pesos de classe e priors) são ajustadas **só no treino de cada fold**.
- **Duas bases**: todos os modelos são avaliados com e sem o atributo `inativo` (`Status`), por suspeita de vazamento. A base **sem `inativo`** é a referência para as escolhas.
- **Ponto de operação**: o cliente é classificado como churn quando a probabilidade prevista é ≥ 0,5. O desbalanceamento é tratado em todos os algoritmos pelo critério "balanced" (pesos de classe, `scale_pos_weight` ou priors uniformes).
- **Escolha das configurações**: em cada notebook, a configuração que representa o algoritmo é a de maior PR-AUC média na base sem `inativo`, com uma regra de parcimônia: entre as que ficam a menos de um desvio da melhor, fica a mais simples.
- **Limitação**: a escolha da configuração e a avaliação usam os **mesmos folds** (não há validação cruzada aninhada). Os números das configurações escolhidas são, por isso, levemente otimistas, mais nos algoritmos com grades maiores (XGBoost, com 12 configurações; regressão logística, com 6).

## Medidas de avaliação e fórmulas

A **matriz de confusão** resume os acertos e erros com a classe churn como positiva:

| | previsto não churn | previsto churn |
|---|---|---|
| **real não churn** | VN (verdadeiro negativo) | FP (falso positivo, falso alarme) |
| **real churn** | FN (falso negativo, churn perdido) | VP (verdadeiro positivo) |

| medida | fórmula | o que mede |
|---|---|---|
| **Recall** (sensibilidade, classe churn) | $\dfrac{VP}{VP + FN}$ | fração dos churns que o modelo encontra. **Métrica principal**: cada FN é um cliente perdido sem ação de retenção |
| **Especificidade** | $\dfrac{VN}{VN + FP}$ | fração dos clientes que ficam e são reconhecidos como tal |
| Precisão (só como parte do F1) | $\dfrac{VP}{VP + FP}$ | fração dos alarmes de churn que estão corretos |
| **F1 macro** | $\dfrac{1}{2}\left(F1_{\text{churn}} + F1_{\text{não churn}}\right)$, com $F1_c = \dfrac{2\,P_c R_c}{P_c + R_c}$ | resumo do desempenho geral, com peso igual para as duas classes, para que a classe churn não seja abafada pela majoritária |
| **Entropia cruzada (log loss)** | $-\dfrac{1}{n}\sum_{i=1}^{n}\left[y_i \log p_i + (1 - y_i)\log(1 - p_i)\right]$ | qualidade das probabilidades; pune fortemente os erros cometidos com alta confiança (quanto menor, melhor) |
| **PR-AUC** (*average precision*) | $\sum_k (R_k - R_{k-1})\, P_k$, percorrendo os limiares em ordem decrescente de probabilidade | área sob a curva precisão-recall: capacidade de pôr os churns no topo da lista, independentemente do limiar. A linha de base (modelo aleatório) é a taxa de churn, **0,156** |

Medidas descartadas, por causa do desbalanceamento (84% de não churn):
- **acurácia**: um modelo que sempre responde "não churn" teria ~84% sem acertar nenhum churn;
- **F1 ponderado**: dá mais peso à classe majoritária;
- **acurácia balanceada**: é a média de recall e especificidade, já analisados separadamente;
- **ROC-AUC**: com muito mais não churn do que churn, a taxa de falsos positivos fica baixa mesmo com muitos falsos alarmes, o que deixa a curva otimista. A PR-AUC foca na classe positiva.

Todas as medidas são calculadas em cada fold de teste e resumidas em **média ± desvio** nos 10 folds. As matrizes de confusão são **somadas** nos 10 folds (cada cliente aparece uma vez).

## Carregamento dos resultados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import precision_recall_curve

from comum import (ALGORITMOS, METRICAS, N_CHURN, N_LINHAS, RESULTS_DIR, carregar_base,
                   diferenca_pareada, formatar_metricas, ler_escolhida, ler_resultados,
                   resultado_salvo, tabela_desbalanceamento)

NOMES = {'mlp': 'MLP', 'nb': 'Naive Bayes', 'xgb': 'XGBoost', 'rl': 'Regressão logística'}
BASES = ['sem-status', 'com-status']
TAXA_CHURN = N_CHURN / N_LINHAS

metricas, matrizes, probas = ler_resultados()
assert (matrizes[['tn', 'fp', 'fn', 'tp']].sum(axis=1) == N_LINHAS).all()
assert (matrizes[['fn', 'tp']].sum(axis=1) == N_CHURN).all()

ESCOLHIDAS = {alg: ler_escolhida(alg) for alg in ALGORITMOS}
for alg, config in ESCOLHIDAS.items():
    for base in BASES:
        assert (alg, config, base) in probas, f'falta {alg} {config} {base}: rode o notebook de modelagem'
print('configurações escolhidas:')
for alg, config in ESCOLHIDAS.items():
    print(f'  {NOMES[alg]:<20} {config}')

# resultados das escolhidas no formato do avaliar: (alg, base) -> {'por_fold', 'matriz'}
res = {(alg, base): resultado_salvo(metricas, matrizes, alg, ESCOLHIDAS[alg], base)
       for alg in ALGORITMOS for base in BASES}

_, y = carregar_base(com_status=True)

## Tabela comparativa

Configuração escolhida de cada algoritmo, nas duas bases: média ± desvio nos 10 folds, e os erros (FN e FP) somados nos folds. As linhas de referência são as regras de um atributo da análise exploratória e o classificador trivial que responde sempre "não churn".

In [ ]:
linhas = []
for base in BASES:
    for alg in ALGORITMOS:
        r = res[(alg, base)]
        (tn, fp), (fn, tp) = r['matriz']
        linhas.append({'base': base, 'algoritmo': NOMES[alg], 'config': ESCOLHIDAS[alg],
                       **formatar_metricas(r['por_fold']), 'FN': fn, 'FP': fp})

referencias = [
    {'base': '—', 'algoritmo': 'regra: reclamou → churn', 'config': '—', 'recall': '0.426', 'especificidade': '0.983'},
    {'base': '—', 'algoritmo': 'regra: inativo → churn', 'config': '—', 'recall': '0.729', 'especificidade': '0.851'},
    {'base': '—', 'algoritmo': 'trivial: sempre não churn', 'config': '—', 'recall': '0.000',
     'especificidade': '1.000', 'pr_auc': f'{TAXA_CHURN:.3f}', 'FN': N_CHURN, 'FP': 0},
]
tabela = pd.DataFrame(linhas + referencias)
tabela.to_csv(RESULTS_DIR / 'comparacao-tabela.csv', index=False)
tabela

### Todas as configurações salvas

Inclui as configurações finais de cada notebook e as versões sem o tratamento do desbalanceamento (`_sem-pesos` e `priors-base`). Serve de apoio à discussão sobre o efeito das parametrizações; as grades completas estão em cada notebook de modelagem.

In [ ]:
completa = []
for (alg, config, base), _ in sorted(probas.items()):
    r = resultado_salvo(metricas, matrizes, alg, config, base)
    (tn, fp), (fn, tp) = r['matriz']
    completa.append({'algoritmo': NOMES[alg], 'config': config, 'base': base,
                     'escolhida': '✔' if config == ESCOLHIDAS[alg] else '',
                     **formatar_metricas(r['por_fold']), 'FN': fn, 'FP': fp})
pd.DataFrame(completa)

## A resposta depende da medida?

Posição de cada algoritmo (1 = melhor) em cada medida, pela média nos 10 folds. No log loss, o menor valor é o melhor; nas demais, o maior.

In [ ]:
medias = pd.DataFrame([{'base': base, 'algoritmo': NOMES[alg], **res[(alg, base)]['por_fold'][METRICAS].mean()}
                       for base in BASES for alg in ALGORITMOS])

fig, eixos = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, base in zip(eixos, BASES):
    sub = medias[medias['base'] == base].set_index('algoritmo')[METRICAS]
    ranks = sub.rank(ascending=False, method='min')
    ranks['log_loss'] = sub['log_loss'].rank(ascending=True, method='min')
    sns.heatmap(ranks.astype(int), annot=sub.round(3).astype(str) + '\n(' + ranks.astype(int).astype(str) + 'º)',
                fmt='', cmap='RdYlGn_r', vmin=1, vmax=4, cbar=False, ax=ax)
    ax.set_title(f'posição por medida — base {base}')
    ax.set_xlabel('')
    ax.set_ylabel('')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'comparacao-ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## Comparação pareada entre os algoritmos

Em cada base, o algoritmo de maior PR-AUC média é a referência. Para cada um dos outros, a diferença (outro − melhor) é calculada fold a fold, o que é possível porque todos usaram os mesmos folds. Se a diferença média for menor que o seu desvio, os dois não se distinguem além da variação entre folds. A última coluna conta em quantos dos 10 folds o melhor teve PR-AUC maior.

In [ ]:
linhas_par = []
for base in BASES:
    sub = medias[medias['base'] == base]
    melhor_alg = [a for a in ALGORITMOS if NOMES[a] == sub.sort_values('pr_auc', ascending=False).iloc[0]['algoritmo']][0]
    ref = res[(melhor_alg, base)]
    for alg in ALGORITMOS:
        if alg == melhor_alg:
            continue
        outro = res[(alg, base)]
        vitorias = int((ref['por_fold']['pr_auc'] > outro['por_fold']['pr_auc']).sum())
        linhas_par.append({'base': base, 'comparação': f'{NOMES[alg]} − {NOMES[melhor_alg]}',
                           **diferenca_pareada(outro, ref),
                           f'folds em que o melhor vence (PR-AUC)': vitorias})
pd.DataFrame(linhas_par)

## Matrizes de confusão

Configuração escolhida de cada algoritmo, somada nos 10 folds.

In [ ]:
fig, eixos = plt.subplots(2, len(ALGORITMOS), figsize=(4.3 * len(ALGORITMOS), 8.5))
for i, base in enumerate(BASES):
    for j, alg in enumerate(ALGORITMOS):
        ax = eixos[i, j]
        sns.heatmap(res[(alg, base)]['matriz'], annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=['não churn', 'churn'], yticklabels=['não churn', 'churn'], ax=ax)
        ax.set_title(f'{NOMES[alg]} ({ESCOLHIDAS[alg]})\nbase {base}')
        ax.set_xlabel('previsto')
        ax.set_ylabel('real')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'comparacao-matrizes.png', dpi=150, bbox_inches='tight')
plt.show()

## Curvas precisão-recall

Curva de cada algoritmo a partir das probabilidades fora do fold (cada cliente com a probabilidade prevista quando estava no teste). O ponto marca o limiar 0,5 usado nas demais medidas, e a linha tracejada é a linha de base (taxa de churn, 0,156).

A curva junta os 10 folds numa só, por isso a área sob ela pode diferir um pouco da média da PR-AUC por fold da tabela.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
for ax, base in zip(eixos, BASES):
    for alg in ALGORITMOS:
        p = probas[(alg, ESCOLHIDAS[alg], base)]
        precisao, recall, _ = precision_recall_curve(y, p)
        linha, = ax.plot(recall, precisao, label=NOMES[alg])
        pred = p >= 0.5
        vp = (pred & (y == 1)).sum()
        ax.plot(vp / (y == 1).sum(), vp / max(pred.sum(), 1), 'o', color=linha.get_color(), ms=8)
    ax.axhline(TAXA_CHURN, color='gray', ls='--', lw=1, label='linha de base')
    ax.set_title(f'curvas precisão-recall — base {base} (ponto: limiar 0,5)')
    ax.set_xlabel('recall')
    ax.grid(alpha=0.3)
eixos[0].set_ylabel('precisão')
eixos[0].legend(loc='lower left')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'comparacao-curvas-pr.png', dpi=150, bbox_inches='tight')
plt.show()

## Comportamento dos erros

Os algoritmos erram nos **mesmos clientes**? Com as probabilidades fora do fold e o limiar 0,5, identificamos os churns perdidos (FN) e os falsos alarmes (FP) de cada algoritmo, na base sem `inativo`. A sobreposição entre dois algoritmos é medida pelo **índice de Jaccard**: |A ∩ B| / |A ∪ B|, de 0 (nenhum erro em comum) a 1 (os mesmos erros).

In [ ]:
BASE_ERROS = 'sem-status'
pred = {alg: probas[(alg, ESCOLHIDAS[alg], BASE_ERROS)] >= 0.5 for alg in ALGORITMOS}
erros_fn = {alg: set(np.where(~pred[alg] & (y == 1))[0]) for alg in ALGORITMOS}
erros_fp = {alg: set(np.where(pred[alg] & (y == 0))[0]) for alg in ALGORITMOS}

# conferência com as matrizes salvas (pode diferir em clientes com probabilidade exatamente 0,5)
for alg in ALGORITMOS:
    (tn, fp), (fn, tp) = res[(alg, BASE_ERROS)]['matriz']
    print(f'{NOMES[alg]:<20} FN {len(erros_fn[alg]):>4} (matriz {fn}) | FP {len(erros_fp[alg]):>4} (matriz {fp})')


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else np.nan


fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
for ax, (titulo, erros) in zip(eixos, [('churns perdidos (FN)', erros_fn), ('falsos alarmes (FP)', erros_fp)]):
    mat = pd.DataFrame([[jaccard(erros[a], erros[b]) for b in ALGORITMOS] for a in ALGORITMOS],
                       index=[NOMES[a] for a in ALGORITMOS], columns=[NOMES[a] for a in ALGORITMOS])
    sns.heatmap(mat, annot=True, fmt='.2f', cmap='Purples', vmin=0, vmax=1, cbar=False, ax=ax)
    ax.set_title(f'sobreposição de {titulo} — Jaccard, base {BASE_ERROS}')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'comparacao-erros-sobreposicao.png', dpi=150, bbox_inches='tight')
plt.show()

### Clientes que todos os algoritmos erram

Clientes classificados errado pelos quatro algoritmos ao mesmo tempo, comparados ao restante da classe real. Se esses clientes têm perfil típico da classe oposta (por exemplo, churns que nunca reclamaram e usam muito o serviço), o erro tende a vir dos dados, e não do algoritmo: ruído de rótulo (a base tem 14 combinações de atributos com churn 0 e 1) ou falta de informação nos atributos.

In [ ]:
X_com, _ = carregar_base(com_status=True)
perfil = pd.DataFrame({
    'reclamou': X_com['reclamacao'],
    'inativo': X_com['inativo'],
    'plano contratual': X_com['plano_contratual'],
    'sem uso': (X_com['qtd_ligacoes'] == 0).astype(int),
    'ligações (mediana)': np.expm1(X_com['qtd_ligacoes']),
    'meses de assinatura (mediana)': X_com['meses_assinatura'],
})

todos_fn = set.intersection(*erros_fn.values())
todos_fp = set.intersection(*erros_fp.values())
print(f'churns perdidos por todos: {len(todos_fn)} de {N_CHURN}')
print(f'falsos alarmes de todos:   {len(todos_fp)} de {N_LINHAS - N_CHURN}')


def resumo_perfil(idx):
    sub = perfil.iloc[sorted(idx)]
    out = sub[['reclamou', 'inativo', 'plano contratual', 'sem uso']].mean().round(3)
    out['ligações (mediana)'] = sub['ligações (mediana)'].median()
    out['meses de assinatura (mediana)'] = sub['meses de assinatura (mediana)'].median()
    out['clientes'] = len(sub)
    return out


churn_idx = set(np.where(y == 1)[0])
fica_idx = set(np.where(y == 0)[0])
pd.DataFrame({
    'churns perdidos por todos': resumo_perfil(todos_fn),
    'demais churns': resumo_perfil(churn_idx - todos_fn),
    'falsos alarmes de todos': resumo_perfil(todos_fp),
    'demais não churns': resumo_perfil(fica_idx - todos_fp),
})

## Efeito do desbalanceamento (consolidado)

Para cada algoritmo, uma configuração com e sem o tratamento do desbalanceamento, comparadas fold a fold:
- MLP: `relu_adam_32` com e sem `sample_weight`;
- Naive Bayes: `completo` (priors 0,5 / 0,5) × `priors-base` (priors do treino);
- XGBoost e regressão logística: a configuração escolhida com e sem `scale_pos_weight` / `class_weight`.

In [ ]:
PARES_DESB = {
    'mlp': ('relu_adam_32', 'relu_adam_32_sem-pesos'),
    'nb': ('completo', 'priors-base'),
    'xgb': (ESCOLHIDAS['xgb'], f'{ESCOLHIDAS["xgb"]}_sem-pesos'),
    'rl': (ESCOLHIDAS['rl'], f'{ESCOLHIDAS["rl"]}_sem-pesos'),
}

linhas_desb, variacao = [], []
for base in BASES:
    for alg, (com_trat, sem_trat) in PARES_DESB.items():
        r_com = resultado_salvo(metricas, matrizes, alg, com_trat, base)
        r_sem = resultado_salvo(metricas, matrizes, alg, sem_trat, base)
        for linha in tabela_desbalanceamento(r_com, r_sem, base):
            linhas_desb.append({'algoritmo': NOMES[alg], **linha})
        dif = r_sem['por_fold'][METRICAS].mean() - r_com['por_fold'][METRICAS].mean()
        variacao.append({'base': base, 'algoritmo': NOMES[alg], **dif})
display(pd.DataFrame(linhas_desb))

variacao = pd.DataFrame(variacao)
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, base in zip(eixos, BASES):
    sub = variacao[variacao['base'] == base].set_index('algoritmo')[['recall', 'especificidade', 'pr_auc']]
    sub.plot.bar(ax=ax, rot=0)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'variação sem − com tratamento — base {base}')
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)
eixos[0].set_ylabel('diferença na média dos folds')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'comparacao-desbalanceamento.png', dpi=150, bbox_inches='tight')
plt.show()

## Efeito do `inativo` (consolidado)

Diferença fold a fold (base com `inativo` − base sem `inativo`) na configuração escolhida de cada algoritmo. Valores positivos indicam que o atributo melhora a métrica, o que, sob a suspeita de vazamento, mede quanto ele "infla" os resultados.

In [ ]:
pd.DataFrame([{'algoritmo': NOMES[alg], 'config': ESCOLHIDAS[alg],
               **diferenca_pareada(res[(alg, 'com-status')], res[(alg, 'sem-status')])}
              for alg in ALGORITMOS])

## Discussão

_A preencher após a execução. Cada pergunta do enunciado e a seção que a responde:_

- **Qual algoritmo apresentou o melhor desempenho? A resposta depende da medida?** → tabela comparativa, "A resposta depende da medida?" e comparação pareada.
- **A alteração dos parâmetros modificou os resultados?** → notebooks de modelagem (grades e comparações pareadas entre configurações) e a tabela de todas as configurações salvas.
- **Os algoritmos apresentaram comportamentos diferentes em relação aos erros?** → matrizes de confusão, FN × FP da tabela comparativa e "Comportamento dos erros".
- **O desbalanceamento influenciou a avaliação?** → "Efeito do desbalanceamento" e a justificativa das medidas (acurácia e ROC-AUC descartadas, linha de base da PR-AUC).
- **Quais características da base explicam as diferenças?** → não linearidades da análise exploratória (idade, faixa de cobrança, falhas × reclamação), atributos redundantes (efeito no NB), clientes sem uso, `inativo` como possível vazamento, ruído de rótulo ("clientes que todos erram").
- **Qual modelo você recomendaria?** → considerar o recall (métrica principal), a especificidade (custo dos falsos alarmes nas ações de retenção), a PR-AUC, a interpretabilidade (coeficientes da regressão logística, importância do XGBoost) e o desempenho na base **sem `inativo`**, que é a mais honesta para prever o churn com 3 meses de antecedência.